# Lab 05 — Autoencoders and latent representations

**Dataset:** [ylecun/mnist](https://huggingface.co/datasets/ylecun/mnist). Train a bottleneck autoencoder, compare latent sizes on validation data, and evaluate the preselected models on an untouched test split.

## The math behind the experiment

An autoencoder has an encoder and decoder:

$$
z=f_\phi(x),\qquad \hat{x}=g_\theta(z).
$$

Here $N$ is the batch size and $D$ is the number of pixels per example. Training minimizes the mean squared error over both examples and pixels, for example

$$
L_{rec}=\frac{1}{ND}\sum_{i=1}^{N}\sum_{j=1}^{D}(x_{ij}-\hat{x}_{ij})^2.
$$

The bottleneck $z$ forces the network to decide what information is worth preserving. A denoising autoencoder makes this more interesting by encoding a corrupted input $\tilde{x}$ while still reconstructing the clean $x$.

In [ ]:
!pip -q install datasets torch pandas matplotlib scikit-learn
from datasets import load_dataset
import numpy as np,pandas as pd,torch
from torch import nn
tr=load_dataset("ylecun/mnist",split="train[:3000]"); te=load_dataset("ylecun/mnist",split="test[:500]")
def arr(ds): return torch.tensor(np.array([np.array(x).reshape(-1) for x in ds["image"]])/255.,dtype=torch.float32)
Xall,Xte=arr(tr),arr(te)
yall=np.asarray(tr["label"]); yte=np.asarray(te["label"])
split_rng=np.random.default_rng(17)
perm=split_rng.permutation(len(Xall))
n_val=max(1,int(.2*len(Xall)))
val_idx,train_idx=perm[:n_val],perm[n_val:]
Xtr,Xval=Xall[train_idx],Xall[val_idx]
ytr,yval=yall[train_idx],yall[val_idx]
torch.manual_seed(0); np.random.seed(0)
class AE(nn.Module):
 def __init__(self,z=16): super().__init__(); self.enc=nn.Sequential(nn.Linear(784,128),nn.ReLU(),nn.Linear(128,z)); self.dec=nn.Sequential(nn.Linear(z,128),nn.ReLU(),nn.Linear(128,784),nn.Sigmoid())
 def forward(self,x): return self.dec(self.enc(x))
def train_ae(noise=0.,z=16):
 m=AE(z); opt=torch.optim.Adam(m.parameters(),1e-3); loss=nn.MSELoss()
 for _ in range(8):
  xb=Xtr+noise*torch.randn_like(Xtr); xb=xb.clamp(0,1); opt.zero_grad(); loss(m(xb),Xtr).backward(); opt.step()
 with torch.no_grad(): rec=m(Xval); mse=loss(rec,Xval).item(); return m,mse
base,base_mse=train_ae(0.); print("validation clean reconstruction MSE",base_mse)

## Step 1 — Prediction

A smaller latent bottleneck should force compression and increase reconstruction error. A denoising objective should make the decoder reconstruct clean images from corrupted inputs, so it may outperform the plain autoencoder on noisy validation inputs.

In [ ]:
denoise,denoise_clean=train_ae(.30); noise=.30*torch.randn_like(Xval); noisy=(Xval+noise).clamp(0,1)
loss=nn.MSELoss()
with torch.no_grad():
 plain_noisy=loss(base(noisy),Xval).item(); denoise_noisy=loss(denoise(noisy),Xval).item()
rows=[["plain",base_mse,plain_noisy],["denoising",denoise_clean,denoise_noisy]]
pd.DataFrame(rows,columns=["model","validation_clean_MSE","validation_noisy_input_MSE"])

## Step 2 — Bottleneck intervention

Change only latent size. Record reconstruction error and parameter count.

In [ ]:
rows=[]
for z in [4,16,64]:
 m,mse=train_ae(0.,z); rows.append([z,sum(p.numel() for p in m.parameters()),mse])
pd.DataFrame(rows,columns=["latent_dim","parameters","validation_reconstruction_MSE"])

## Visualize the representation — reconstruction and bottleneck size

First compare how the bottleneck dimension changes validation reconstruction error. Then inspect validation images as input, noisy input, and reconstructions. The picture helps identify whether the model preserves recognizable structure or merely optimizes average pixel error.

In [ ]:
import matplotlib.pyplot as plt

latent_results = pd.DataFrame(rows, columns=["latent_dim", "parameters", "validation_reconstruction_MSE"])
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(latent_results["latent_dim"], latent_results["validation_reconstruction_MSE"], marker="o")
ax.set_title("Bottleneck size vs reconstruction error")
ax.set_xlabel("Latent dimension")
ax.set_ylabel("Validation reconstruction MSE")
ax.grid(alpha=.25)
fig.tight_layout()
plt.show()

with torch.no_grad():
    clean = Xval[:6]
    noisy_sample = (clean + .30 * torch.randn_like(clean)).clamp(0, 1)
    plain_rec = base(noisy_sample)
    denoise_rec = denoise(noisy_sample)
images = [clean, noisy_sample, plain_rec, denoise_rec]
titles = ["Clean target", "Noisy input", "Plain AE", "Denoising AE"]
fig2, axs = plt.subplots(4, 6, figsize=(10, 6))
for r, (batch, title) in enumerate(zip(images, titles)):
    for cidx in range(6):
        axs[r, cidx].imshow(batch[cidx].reshape(28, 28).cpu().numpy(), cmap="gray", vmin=0, vmax=1)
        axs[r, cidx].axis("off")
    axs[r, 0].set_ylabel(title, rotation=0, labelpad=55, va="center")
fig2.suptitle("What does the autoencoder reconstruct?")
fig2.tight_layout()
plt.show()

## Step 3 — Test the representation with a frozen-encoder linear probe

Reconstruction error asks whether the decoder can rebuild the input. It does not tell us directly whether the latent code separates classes. Freeze the trained encoder, extract its latent vectors, and train only a logistic-regression classifier on those vectors. The encoder's weights are not updated during this step.

For context, compare against a classifier on raw pixels and a classifier on features from an untrained random encoder of the same latent size. The random baseline helps test whether the trained encoder learned a useful representation rather than merely providing a dimensionality reduction. This is a small linear-probe diagnostic, not a definitive semantic benchmark.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

torch.manual_seed(123)
random_encoder=AE(z=16)
with torch.no_grad():
    z_train=base.enc(Xtr).cpu().numpy()
    z_val=base.enc(Xval).cpu().numpy()
    zr_train=random_encoder.enc(Xtr).cpu().numpy()
    zr_val=random_encoder.enc(Xval).cpu().numpy()

y_train=ytr
y_val=yval
feature_sets=[
    ("raw pixels",Xtr.cpu().numpy(),Xval.cpu().numpy()),
    ("trained AE encoder (frozen)",z_train,z_val),
    ("random encoder (frozen)",zr_train,zr_val)
]
probe_rows=[]
for name,features_train,features_val in feature_sets:
    probe=make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=500,solver="lbfgs",random_state=0)
    )
    probe.fit(features_train,y_train)
    prediction=probe.predict(features_val)
    probe_rows.append([
        name,
        features_train.shape[1],
        accuracy_score(y_val,prediction),
        f1_score(y_val,prediction,average="macro")
    ])
probe_results=pd.DataFrame(
    probe_rows,
    columns=["features","feature_dimensions","validation_accuracy","validation_macro_F1"]
)
probe_results

### Interpret the linear probe

The encoder is frozen: only the logistic-regression head learns from labels. If the trained autoencoder's features outperform the random-encoder features, that is evidence that reconstruction training organized information in a way a linear classifier can use. If raw pixels perform better, compression may have discarded class-relevant detail. If all results are close, inspect the sample size, latent size, training budget, and uncertainty before drawing a strong conclusion.

This comparison uses the same train/validation examples and classifier family, but feature dimensions differ; report that fact rather than calling it a perfectly matched-capacity benchmark.

## Final check on the untouched test split

Choose the latent size using validation reconstruction error. The following cell refits the selected plain autoencoder on all training examples, then reports test reconstruction quality. It also performs the predeclared raw-pixel / trained-encoder / random-encoder linear-probe comparison on the test split after fitting each probe on all training labels. Do not tune the model based on these final test results.


In [ ]:
best_z=int(latent_results.loc[latent_results["validation_reconstruction_MSE"].idxmin(),"latent_dim"])
def train_ae_final(data,noise=0.0,z=16,seed=101):
    torch.manual_seed(seed)
    model=AE(z); opt=torch.optim.Adam(model.parameters(),lr=1e-3); loss_fn=nn.MSELoss()
    for _ in range(8):
        corrupted=(data+noise*torch.randn_like(data)).clamp(0,1)
        opt.zero_grad(); loss_fn(model(corrupted),data).backward(); opt.step()
    return model
final_plain=train_ae_final(Xall,noise=0.0,z=best_z,seed=101)
final_denoise=train_ae_final(Xall,noise=.30,z=16,seed=102)
noise_test=.30*torch.randn_like(Xte)
noisy_test=(Xte+noise_test).clamp(0,1)
loss_fn=nn.MSELoss()
with torch.no_grad():
    reconstruction_rows=[
        ["plain selected bottleneck",best_z,float(loss_fn(final_plain(Xte),Xte).item()),float(loss_fn(final_plain(noisy_test),Xte).item())],
        ["denoising bottleneck 16",16,float(loss_fn(final_denoise(Xte),Xte).item()),float(loss_fn(final_denoise(noisy_test),Xte).item())]
    ]
final_reconstruction=pd.DataFrame(reconstruction_rows,columns=["model","latent_dim","test_clean_MSE","test_noisy_input_MSE"])
torch.manual_seed(123)
random_final=AE(z=best_z)
def features(model,data):
    with torch.no_grad(): return model.enc(data).cpu().numpy()
probe_sets=[
    ("raw pixels",Xall.cpu().numpy(),Xte.cpu().numpy()),
    ("trained AE encoder (frozen)",features(final_plain,Xall),features(final_plain,Xte)),
    ("random encoder (frozen)",features(random_final,Xall),features(random_final,Xte))
]
probe_rows=[]
for name,train_features,test_features in probe_sets:
    probe=make_pipeline(StandardScaler(),LogisticRegression(max_iter=500,solver="lbfgs",random_state=0))
    probe.fit(train_features,yall)
    pred=probe.predict(test_features)
    probe_rows.append([name,train_features.shape[1],accuracy_score(yte,pred),f1_score(yte,pred,average="macro")])
final_probe=pd.DataFrame(probe_rows,columns=["features","feature_dimensions","test_accuracy","test_macro_F1"])
display(final_reconstruction)
display(final_probe)


## Conclusion / answer key

Expected: smaller bottlenecks usually increase reconstruction error; denoising training should reduce sensitivity to input corruption when the noise level matches the training intervention. A low reconstruction error does not automatically mean the latent space is useful for every downstream task.

### Research extension
Freeze encoder representations and test a linear classifier. Compare representation quality against reconstruction quality.